**Indexing**
1. Load the pdf data using PyMyPDFLoader from langchain_community.document_loaders
2. Split the text/docs into chunks using Recursive... from langchain_textsplitters
3. Generate text embeddings with:
    - OpenAI text-embedding-3-small
    - Testing with ollama:
      Embedding model: ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
      Language model: ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF
4. Store embeddings in a vector database:
    - Pinecone
    - FAISS
    - Chromadb

**Retrieval**
1. Translate users question to embeddings
2. Retrieve relevant chunks according to the question as context
3. Add a system prompt (a prompt for the llm's behavior), the context and the user's question (again)
4. Push this result to the llm to generate a grounded answer


In [8]:
import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec

# Load keys from a .env file 
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME", "knust-admission-rag")
SPARSE_INDEX_NAME = os.getenv("SPARSE_INDEX_NAME", "knust-rag-sparse")

# Initialize clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)
pinecone_client = Pinecone(api_key=PINECONE_API_KEY)

### Load PDF

In [9]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(file_path="./data/admission_requirement.pdf")
docs = loader.load()
print(docs)

Performing OCR on page.number=57[58]...
[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2026-08-03T18:57:14+00:00', 'source': './data/admission_requirement.pdf', 'file_path': './data/admission_requirement.pdf', 'total_pages': 58, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-03T18:57:53+00:00', 'trapped': '', 'modDate': 'D:20260803185753Z', 'creationDate': 'D:20260803185714Z', 'page': 0}, page_content='**KNUST Kwame Nkrumah** University of Science and Technology \n\nENTRY REQUIREMENTS AND GUIDELINES FOR SELECTING AN UNDERGRADUATE PROGRAMME \n\n'), Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2026-08-03T18:57:14+00:00', 'source': './data/admission_requirement.pdf', 'file_path': './data/admission_requirement.pdf', 'total_pages': 58, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '',

### Split PDF into chunks

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc_splitter = RecursiveCharacterTextSplitter(
  chunk_size=500,
  chunk_overlap=100
)
chunked_docs = doc_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunked_docs)}")

for i, chunk in enumerate(chunked_docs, start=1):
  print(f"\nChunk {i}:")
  print(f"metadata: {chunk.metadata}\n")
  print(f"content: '{chunk.page_content[:150]}'")


Number of chunks: 287

Chunk 1:
metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2026-08-03T18:57:14+00:00', 'source': './data/admission_requirement.pdf', 'file_path': './data/admission_requirement.pdf', 'total_pages': 58, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-03T18:57:53+00:00', 'trapped': '', 'modDate': 'D:20260803185753Z', 'creationDate': 'D:20260803185714Z', 'page': 0}

content: '**KNUST Kwame Nkrumah** University of Science and Technology 

ENTRY REQUIREMENTS AND GUIDELINES FOR SELECTING AN UNDERGRADUATE PROGRAMME'

Chunk 2:
metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2026-08-03T18:57:14+00:00', 'source': './data/admission_requirement.pdf', 'file_path': './data/admission_requirement.pdf', 'total_pages': 58, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': 

### Translate Chunks into embeddings

In [17]:
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(
  model="text-embedding-3-small"
)

In [18]:
from langchain_chroma import Chroma

chroma_vector_db = Chroma(
  embedding_function=embedding_function,
  persist_directory="./chroma_vector_db"
)
print(f"Chroma Vector Database with {chroma_vector_db._collection.count()} embeddings")


Chroma Vector Database with 287 embeddings


### Retrieval

In [19]:
# Store into vectordb
retriever = chroma_vector_db.as_retriever(
  search_kwargs={"k":2}
)

retrieved_docs = retriever.invoke("What is the cutoff point Sociology")
print(retrieved_docs)

[Document(id='9b5220bb-01e6-4f49-bf4a-26901166e1aa', metadata={'file_path': './data/admission_requirement.pdf', 'page': 55, 'keywords': '', 'trapped': '', 'creationDate': 'D:20260803185714Z', 'total_pages': 58, 'producer': 'Adobe PDF Library 17.0', 'title': '', 'moddate': '2026-08-03T18:57:53+00:00', 'format': 'PDF 1.4', 'author': '', 'subject': '', 'source': './data/admission_requirement.pdf', 'creator': 'Adobe InDesign 18.1 (Windows)', 'modDate': 'D:20260803185753Z', 'creationdate': '2026-08-03T18:57:14+00:00'}, page_content='## **CUT-OFF PROGRAM AGGREGATE** \n\n## **CUT-OFF PROGRAM AGGREGATE** \n\n## **COLLEGE OF HEALTH SCIENCES** \n\n## **COLLEGE OF HUMANITIES AND SOCIAL SCIENCES**'), Document(id='c2ae4fba-5a79-4121-a684-da9c0349e19d', metadata={'source': './data/admission_requirement.pdf', 'creationDate': 'D:20260803185714Z', 'creationdate': '2026-08-03T18:57:14+00:00', 'creator': 'Adobe InDesign 18.1 (Windows)', 'page': 53, 'producer': 'Adobe PDF Library 17.0', 'moddate': '2026-0

### Generation

In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
  model="gpt-4o-mini"
)

In [23]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are KNUST Admissions Assistant, an AI agent that helps prospective students find 
admission requirements and information for programs at Kwame Nkrumah University of 
Science and Technology (KNUST).

## Your Role
- Help users identify the correct program based on the subjects/field they want to study.
- Provide the entry requirements for that program, strictly using 
  the retrieved context provided to you.
- Clarify ambiguous program names before answering (e.g. "BSc. Computer Science" vs 
  "BSc. Computer Engineering" vs "Information Technology").

## Grounding Rules
- Only answer using information present in the retrieved context. Never rely on prior 
  knowledge of KNUST cutoffs, as these change every admission cycle.
- If the retrieved context does not contain the answer, say so plainly and suggest the 
  user check the official KNUST admissions portal or contact the admissions office — 
  do not guess or estimate a cutoff point.
- If multiple years of cutoff data appear in the context, default to the most recent 
  year unless the user asks otherwise, and state which year you're quoting.
- Always distinguish between aggregate cutoff points for different qualification tracks 
  if the context includes more than one (e.g. WASSCE vs SSSCE, or Arts vs Science 
  aggregate systems), and ask the user which applies to them if unclear.

## Handling Ambiguity
- If a user names a subject area rather than a specific program (e.g. "I want to study 
  medicine-related courses"), list the matching programs from the retrieved context and 
  ask which one they mean before giving requirements.
- If a program name is misspelled or informally phrased, match it to the closest known 
  program in the context and confirm with the user.

## Response Format
- State the program name clearly.
- Give the cutoff point (and the year it applies to).
- List core entry requirements (e.g. required WASSCE core/elective subjects, grade 
  thresholds).
- Note any additional requirements (interviews, portfolios, quotas) if present in context.
- Keep responses concise and structured — use short lists over long paragraphs.

## Tone
- Warm, encouraging, and clear — many users are first-time applicants or parents. 
  Avoid jargon; explain grading systems (e.g. aggregate scoring) briefly if relevant.

## Boundaries
- Do not provide admission advice outside KNUST (e.g. other universities) unless asked 
  for a comparison, and even then, only using retrieved context.
- Do not fabricate cutoff numbers, subject combinations, or deadlines under any 
  circumstances. When context is insufficient, be transparent about the gap."""

prompt = ChatPromptTemplate.from_messages([
      ("system", system_prompt + "\n\nRetrived context:\n{context}"),
      ("human", "{question}"),
  ])

# prompt = ChatPromptTemplate.from_template(system_prompt)
# print(prompt)

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm |StrOutputParser()
)

response = chain.invoke("What's the requirement for reading Mathematics")
print(response)

The program you are referring to is **BSc. Mathematics**. 

### Entry Requirements:

#### A. WASSCE/SSSCE Applicants
- **Core Subjects**: Credit passes in English Language, Mathematics, and Integrated Science.
- **Elective Subjects**: Credit passes in Mathematics, Physics, and Chemistry or Biology.

#### B. ‘A’ Level and Equivalent Applicants
- **GCE ‘O’ Level**: Credit passes in FIVE (5) subjects, including English Language and Mathematics.
- **GCE ‘A’ Level**: Credit passes in THREE (3) subjects, including Mathematics.

#### C. Mature Applicants
- Must be at least 25 years old at the time of application.
- Must satisfy the general requirements for the program (i.e., WASSCE/SSSCE, or GCE ‘O’ and ‘A’ Levels, or HND/Diploma).

If you have any further questions or need clarification, feel free to ask!
